In [1]:
## conda env!!!

# 1. Get Data. 
## 1.1 Get Alleles from config

In [55]:
from config import CLASSES_TO_USE, ORGANISMS_TO_USE, PER_ORGANISM_PER_CLASS_ALLELES

#rewrite classes to use to only I class because it's only class I method
METHOD = 'bertmhc'
ORGANISMS_TO_USE = ['Human']
CLASSES_TO_USE = ['I', 'II']
ALLELES = [item  for organism_to_use in ORGANISMS_TO_USE 
                 for class_to_use in CLASSES_TO_USE 
                 for item in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]]
ALLELES

['HLA_A_02_01',
 'HLA_B_27_05',
 'HLA_A_03_01',
 'HLA_B_07_02',
 'HLA_DRB1_01_01',
 'HLA_DRB1_04_01',
 'HLA_DRB1_04_05']

## 1.2 Read Data files. 
* add id column
* add mhc column
* filter by length if needed

In [56]:
# importing the predict module
from mhcnames import normalize_allele_name## ONLY FOR LINUX

import pandas as pd
import os
import sys
import numpy as np


In [57]:
# Read datasets
DECOY_KEY = 'x1'

DATA_PATH = "/home/denklewer/Data/"
INPUT_DATA_PATH = DATA_PATH + 'input/'
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + 'predictions/'+ METHOD+'/'
per_allele_val_df = {
    allele_name :pd.read_csv("{}df_{}_{}_val_for_classification.csv".format(INPUT_DATA_PATH,allele_name,DECOY_KEY), index_col=0) for allele_name in ALLELES
}
per_allele_train_df = {
    allele_name :pd.read_csv("{}df_{}_{}_train_for_classification.csv".format(INPUT_DATA_PATH,allele_name,DECOY_KEY), index_col=0) for allele_name in ALLELES
}
per_allele_test_df = {
    allele_name :pd.read_csv("{}df_{}_{}_test_for_classification.csv".format(INPUT_DATA_PATH,allele_name,DECOY_KEY), index_col=0) for allele_name in ALLELES
}

In [58]:
def defineClass(allele_name):
    for organism_to_use in ORGANISMS_TO_USE:
        for class_to_use in CLASSES_TO_USE:
            if allele_name in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]:
                return class_to_use

In [59]:
def defineAlleleName(allele_name):
    ## HARDCODE
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    elif allele_name == 'HLA_DRB1_01_01':
        target_allele_name = 'HLA-DRB1*01:01'
    elif allele_name == 'HLA_DRB1_04_01':
        target_allele_name = 'HLA-DRB1*04:01'
    elif allele_name == 'HLA_DRB1_04_05':
        target_allele_name = 'HLA-DRB1*04:05'    
    else:
        target_allele_name = normalize_allele_name(allele_name)
    return target_allele_name

In [60]:
def add_aux_columns(per_allele_df, columns_to_add={'source_sequence_name', 'mhc_allele', 'length'}):
    for allele_name in per_allele_df.keys():
        if 'source_sequence_name' in columns_to_add:
            per_allele_df[allele_name]['source_sequence_name'] =  'seq' + per_allele_df[allele_name].index.astype(str).values
        if 'mhc_allele' in columns_to_add:
            target_allele = defineAlleleName(allele_name)
            per_allele_df[allele_name]['allele'] = target_allele
        if 'length' in columns_to_add:
            per_allele_df[allele_name]['length'] = per_allele_val_df[allele_name]['peptide'].str.len()
            
            
def filter_by_length(per_allele_df, length_bound_class_1=15, length_bound_class_2=30):
    if defineClass(allele_name) == 'I':
        per_allele_val_df[allele_name] = per_allele_val_df[allele_name][per_allele_val_df[allele_name]['length'] <= 15]
    else:
        per_allele_val_df[allele_name] = per_allele_val_df[allele_name][per_allele_val_df[allele_name]['length'] <= 30]
    


            
add_aux_columns(per_allele_val_df)
add_aux_columns(per_allele_train_df)
add_aux_columns(per_allele_test_df)



In [61]:
[defineAlleleName(allele_name) for allele_name in ALLELES]

['HLA-A*02:01',
 'HLA-B*27:05',
 'HLA-A*03:01',
 'HLA-B*07:02',
 'HLA-DRB1*01:01',
 'HLA-DRB1*04:01',
 'HLA-DRB1*04:05']

In [62]:
per_allele_val_df[ALLELES[0]].head()

,peptide,binder,source_sequence_name,allele,length
ID,,,,,
0,KLEPIWNEVGLE,1,seq0,HLA-A*02:01,12
1,ASFVEVVSLW,1,seq1,HLA-A*02:01,10
2,GSLDKTSP,1,seq2,HLA-A*02:01,8
3,KLDDTVYLRDIA,1,seq3,HLA-A*02:01,12
4,SFPFWNLTG,1,seq4,HLA-A*02:01,9


# 2. Preparation. (Add mhc sequence from mhcattnnet)

In [63]:
MHCATTNNET_HOME = os.getenv('MHCATTNNET_HOME')
sys.path.append(MHCATTNNET_HOME)

from src.alleles_I import c1_map
from src.alleles_II import c2_map
MHCATTNNET_HOME

'/home/denklewer/Projects/MHCAttnNet'

## 2.1 Preprocess data
* generate tran-test-val (peptide,mhc_acids, bind, mhc_allele)

In [64]:
def add_mhc_sequence_column(per_allele_df):
    for allele_name, df in per_allele_df.items():
        if defineClass(allele_name) == 'I':
            df['mhc'] = df.allele.apply(lambda x: c1_map.get(x, x))
        else:
            df['mhc'] =  df.allele.apply(lambda x: c2_map.get(x, x))

            
def rename_column(per_allele_df, initial_name, target_name):
    for allele_name, df in per_allele_df.items():
        df.rename(columns={initial_name: target_name}, inplace=True)
            
add_mhc_sequence_column(per_allele_train_df)
add_mhc_sequence_column(per_allele_test_df)
add_mhc_sequence_column(per_allele_val_df)


rename_column(per_allele_train_df, 'binder', 'label')
rename_column(per_allele_test_df,'binder', 'label')
rename_column(per_allele_val_df, 'binder', 'label')


# # Join dfs by class
# def get_per_class_dfs(per_allele_dfs):
#     per_class_df = dict()
#     for class_to_use in CLASSES_TO_USE:
#         class_dfs_list = [per_allele_dfs[allele_name]  for allele_name in per_allele_dfs.keys() if defineClass(allele_name) == class_to_use ]
#         class_result_df = pd.concat(class_dfs_list)
#         per_class_df[class_to_use] = class_result_df
#     return per_class_df

# per_class_train_df = get_per_class_dfs(per_allele_train_df)
# per_class_test_df  = get_per_class_dfs(per_allele_test_df)
# per_class_val_df  = get_per_class_dfs(per_allele_val_df)

# merge all datasets
train_df = pd.concat(list(per_allele_train_df.values()))
val_df = pd.concat(list(per_allele_val_df.values()))



## 2.2 Save Preprocessed files to TMP

In [65]:
os.getcwd()

'/home/denklewer/Projects/MHC_predictor'

In [66]:
# PATH_TO_DATA = f'{PREDICTOR_HOME}/data'
BERTMHC_HOME = os.getenv('BERTMHC_HOME')

TEMPDIR = f'{os.getcwd()}/TMP/'

train_df.to_csv("{}bertmhc_data_train.csv".format(TEMPDIR),index=False )
val_df.to_csv("{}bertmhc_data_val.csv".format(TEMPDIR),index=False)

for allele_name in ALLELES:
    per_allele_val_df[allele_name].to_csv("{}bertmhc_data_val{}_{}".format(TEMPDIR,allele_name,DECOY_KEY), sep='\t',index=False, header=False)


## 2.3 Train bert mhc using training_files

In [44]:
#use console

In [67]:
sys.path.append(BERTMHC_HOME)

In [72]:
from bertmhc.utils_model import train, evaluate
from bertmhc.dataloader import BertDataset
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from bertmhc.bertmhc import BERTMHC
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tape import ProteinBertConfig

In [ ]:
trainset = BertDataset( f'{TEMPDIR}bertmhc_data_train.csv',
                       max_pep_len=30)
valset = BertDataset(f'{TEMPDIR}bertmhc_data_val.csv',
                     max_pep_len=30)

train_data = DataLoader(trainset,
                        batch_size=32,
                        num_workers=16,
                        pin_memory=False,
                        collate_fn=trainset.collate_fn)

val_data = DataLoader(valset,
                      batch_size=32,
                      num_workers=16,
                      pin_memory=False,
                      collate_fn=valset.collate_fn)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BERTMHC.from_pretrained('bert-base')
model = model.to(device)
for p in model.bert.parameters():
    p.requires_grad = False
    
aff_criterion = nn.BCEWithLogitsLoss()
mass_criterion = nn.BCEWithLogitsLoss(reduction='none')
alpha = 0.5
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, 'max', patience=2, min_lr=1e-4, factor=0.1)

for epoch in range(1, 10):
    train(model, optimizer, train_data, device, aff_criterion, mass_criterion, alpha, scheduler)
torch.save(model.state_dict(), f'{TEMPDIR}bert_model.pt')

## 2.4 Evaluate trained model

In [ ]:
config = ProteinBertConfig.from_pretrained('bert-base')
model = BERTMHC(config)
model = model.to(device)
valset = BertDataset(f'{TEMPDIR}bertmhc_data_val.csv',
                     max_pep_len=30)
val_data = DataLoader(valset,
                      batch_size=32,
                      num_workers=16,
                      pin_memory=True,
                          collate_fn=valset.collate_fn)

pred = []
label = []
for batch in val_data:
        batch = {name: tensor.to(device)
                 for name, tensor in batch.items()}
        logits, targets = model(**batch)
        pred.append(torch.sigmoid(logits).cpu().detach().numpy())
        label.append(targets.cpu().detach().numpy())


In [15]:
from src import config
import torch

/home/denklewer/Tools/anacoda3/envs/mhcattnnet/lib/python3.6/site-packages/torch/cuda/__init__.py:52: UserWarning: CUDA initialization: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx (Triggered internally at  /pytorch/c10/cuda/CUDAFunctions.cpp:100.)
  return torch._C._cuda_getDeviceCount() > 0


In [16]:
from src.model import MHCAttnNet
from src.data_loader import DataLoader

In [28]:
#for class 1 
def run_validation_for_class(class_to_use):
    config.read_all_columns = True
    if class_to_use == 'I':
        config.model_name = config.mhc_1_model_name
        config.base_path = config.mhc_1_base_path
    else:
        config.model_name = config.mhc_2_model_name
        config.base_path = config.mhc_2_base_path
    device = config.device
    data_loader_obj  = DataLoader(config)
    train_loader, val_loader, test_loader, peptide_embedding, mhc_embedding = data_loader_obj.get_dataset(device, shuffle=False)
    ## Read model
    model = MHCAttnNet(peptide_embedding, mhc_embedding, config)
    model.load_state_dict(torch.load(config.model_name))
    model.to(device)
    y_actual_val = []
    y_pred_val = []
    seq_id = []
    mhc_allele = []
    for row in tqdm(val_loader):
        result = model.forward(row.peptide,row.mhc_amino_acid)
        y_actual = row.bind
        y_actual_val += list(y_actual.cpu().data.numpy())
        result_idx = torch.max(result, dim=1)[1]
        y_pred_val += list(result_idx.cpu().data.numpy())
        seq_id += list(row.source_sequence_name)
        mhc_allele += list(row.mhc_allele)
    return pd.DataFrame({"source_sequence_name":seq_id, "binder":y_actual_val, "score": y_pred_val, "mhc_allele":mhc_allele})


In [29]:
for class_to_use in CLASSES_TO_USE:
     per_class_val_df[class_to_use] =  run_validation_for_class(class_to_use)

/home/denklewer/Tools/anacoda3/envs/mhcattnnet/lib/python3.6/site-packages/torchtext/data/field.py:150: UserWarning: Field class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.
  warnings.warn('{} class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.'.format(self.__class__.__name__), UserWarning)
/home/denklewer/Tools/anacoda3/envs/mhcattnnet/lib/python3.6/site-packages/torchtext/data/field.py:36: UserWarning: RawField class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.
  warnings.warn('{} class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.'.format(self.__class__.__name__), UserWarning)
/home/denklewer/Tools/anacoda3/envs/mhcattnnet/lib/python3.6/site-packages/torchtext/data/example.py:68: UserWarning: Exam

Peptide embedding dimension torch.Size([22, 100])
MHC Amino Acid embedding dimension torch.Size([136, 100])


  0%|          | 1/229 [00:00<00:41,  5.54it/s]

Peptide embedding dimension torch.Size([22, 100])
MHC Amino Acid embedding dimension torch.Size([88, 100])


100%|██████████| 229/229 [00:38<00:00,  5.90it/s]


## View results

In [30]:
per_class_val_df[CLASSES_TO_USE[0]]

,source_sequence_name,binder,score,mhc_allele
0,seq1855,0,1,HLA-A*02:01
1,seq2260,0,0,HLA-A*02:01
2,seq2340,0,1,HLA-A*02:01
3,seq2564,0,1,HLA-A*02:01
4,seq1942,0,1,HLA-A*02:01
...,...,...,...,...
12231,seq1759,0,1,HLA-B*27:05
12232,seq1886,0,0,HLA-B*27:05
12233,seq2207,0,0,HLA-B*27:05
12234,seq2670,0,1,HLA-B*27:05


## Split by allele

In [38]:
per_allele_predictions = dict()
for allele_name in ALLELES:
    target_allele_name = defineAlleleName(allele_name)
    class_of_allele = defineClass(allele_name)
    
    df = per_class_val_df[class_of_allele]
    df = df[df['mhc_allele'] == target_allele_name]
    per_allele_predictions[allele_name] = per_allele_val_df[allele_name].merge( df[['source_sequence_name', 'score']] , on=['source_sequence_name'])

rename_column(per_allele_predictions, 'bind', 'binder')    
    

#### write to files

In [41]:
# for decoy_key in DECOY_PREFIXES:
for allele_name, total_df in per_allele_predictions.items():
    total_df.to_csv("{}{}_predictions_{}_{}".format(TARGET_PATH_TO_PREDICTIONS,METHOD,allele_name,DECOY_KEY))

In [42]:
total_df

,peptide,binder,source_sequence_name,mhc_allele,length,mhc_amino_acid,score
0,MHSAEKVPLQNPPIKWAEGH,1,seq0,HLA-DRB1*04:05,20,RFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGEYRAVT...,0
1,VRSITPNEKQPE,1,seq1,HLA-DRB1*04:05,12,RFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGEYRAVT...,0
2,RAASLCLPRYSFRRLD,1,seq2,HLA-DRB1*04:05,16,RFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGEYRAVT...,0
3,PGPGQLTLRLRACGLNF,1,seq3,HLA-DRB1*04:05,17,RFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGEYRAVT...,0
4,KELQVVAWDILLNIIERLLQQLQT,1,seq4,HLA-DRB1*04:05,24,RFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGEYRAVT...,0
...,...,...,...,...,...,...,...
2056,APNTFKTLDSWRDEFL,0,seq2056,HLA-DRB1*04:05,16,RFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGEYRAVT...,0
2057,IEFRFYKEITNVFRG,0,seq2057,HLA-DRB1*04:05,15,RFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGEYRAVT...,0
2058,LDGTIEYMRDPDDQ,0,seq2058,HLA-DRB1*04:05,14,RFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGEYRAVT...,0
2059,EVRKLVALKTNGDGNC,0,seq2059,HLA-DRB1*04:05,16,RFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGEYRAVT...,0
